# Sequential Forecasting: Monthly Spend Prediction

Model: Prophet (per user) with Moving Average fallback

- Total spend: Prophet or moving average
- Spend per category: moving average
- Number of transactions: prophet or moving average
- Average transaction amount: derived from total/count

**Strategy**

'>= 4 monthly data points - Prophet

2-3 monthly data point - Weighter MOving Average

< 2 monthly data points - Global segment median

In [1]:
import pandas as pd
import numpy as np
from prophet import Prophet

/workspaces/python/env/featuretools-env/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Importing plotly failed. Interactive plots will not work.


In [2]:
transactions_df = pd.read_csv('../features/transactions_enriched.csv', parse_dates=['date'])
fm_encoded = pd.read_csv('../features/fm_encoded.csv', index_col='user_id')

## Monthly Aggregation

Aggregate raw transactions into monthly total per user before fitting for Prophet as it works on time series.
- Monthly totals per user

In [3]:
transactions_df['month'] = transactions_df['date'].dt.to_period('M').dt.to_timestamp()
transactions_df['amount_abs'] = transactions_df['amount'].abs()

monthly_user = (transactions_df.groupby(['user_id', 'month']).agg(total_spend = ('amount_abs', 'sum'),transaction_count = ('amount_abs', 'count'),avg_amount = ('amount_abs', 'mean'),).reset_index())

monthly_category = (transactions_df.groupby(['user_id', 'month', 'category_id'])['amount_abs'].sum().reset_index().rename(columns={'amount_abs': 'category_spend'}))

segment_map = fm_encoded['task_segment'].to_dict()
monthly_user['segment'] = monthly_user['user_id'].map(segment_map)

months_per_user = monthly_user.groupby('user_id')['month'].count()

print(f'Total user-months: {len(monthly_user)}')
print(f'Monthly data points per user | Min: {months_per_user.min()} | Median: {months_per_user.median()} | Max: {months_per_user.max()}')
print(f'\nProphet (>=4 months): {(months_per_user >= 4).sum()} users')
print(f'Moving Average (2-3 months): {((months_per_user >= 2) & (months_per_user < 4)).sum()} users')
print(f'Segment median (<2 months): {(months_per_user < 2).sum()} users')

Total user-months: 5880
Monthly data points per user | Min: 1 | Median: 22.0 | Max: 28

Prophet (>=4 months): 279 users
Moving Average (2-3 months): 21 users
Segment median (<2 months): 6 users


## Compute Segment Medians

Users with too little history fall back to the median of their `task_segment` group — this is better than a global average since spending patterns differ significantly across segments.

In [4]:
segment_medians = (monthly_user.groupby('segment')[['total_spend', 'transaction_count', 'avg_amount']].median()
)

global_medians = monthly_user[['total_spend', 'transaction_count', 'avg_amount']].median()

print(segment_medians.round(2))

                     total_spend  transaction_count  avg_amount
segment                                                        
Consistent Weekly         373.41                2.0      167.51
Healthy Active User      1177.36                4.0      456.26
High-Intensity User      2187.56               66.0       30.90
Low Activity/Trial       2252.09               26.5       82.53
Single-Tasker             541.95                2.0      170.15


## Forecasting Functions

1. **Prophet** — handles trend + seasonality, built for short irregular series

Fits Prophet to a user's monthly time series.

2. **Weighted Moving Average** — recent months weighted higher (3:2:1)

Weighted Moving Average — most recent month gets highest weight.

Weights: [1, 2, 3] for oldest -> newest.

±20% confidence band


3. **Segment Median** — when there's simply not enough data to trend

Returns the first day of the month after the last data point.

In [7]:
def forecast_with_prophet(series_df, target_col, periods=1):
    df = series_df[['month', target_col]].copy()
    df.columns = ['ds', 'y']
    df = df.dropna().sort_values('ds')

    m = Prophet(
        yearly_seasonality=False,   
        weekly_seasonality=False,  
        daily_seasonality=False,
        changepoint_prior_scale=0.1,  
        interval_width=0.80           
    )
    m.fit(df)

    future = m.make_future_dataframe(periods=periods, freq='MS')
    forecast = m.predict(future)
    last_row = forecast.iloc[-1]

    return (
        max(0, round(float(last_row['yhat']),     2)),
        max(0, round(float(last_row['yhat_lower']), 2)),
        max(0, round(float(last_row['yhat_upper']), 2)),
        last_row['ds']
    )


def forecast_with_wma(series, n_weights=3):
    values = series.dropna().values[-n_weights:]
    if len(values) == 0:
        return None, None, None

    weights  = np.arange(1, len(values) + 1, dtype=float)
    forecast = float(np.average(values, weights=weights))

    lower = max(0, round(forecast * 0.80, 2))
    upper = round(forecast * 1.20, 2)
    return round(forecast, 2), lower, upper

def get_next_month(monthly_df):
    last = monthly_df['month'].max()
    return last + pd.DateOffset(months=1)

## Forecast Category Spend

Category-level forecasting uses WMA only — there are too few monthly data points per category to fit Prophet reliably. For each user we forecast their top 3 categories by spend.

In [8]:
def forecast_categories(user_id, monthly_category, top_n=3):
    user_cat = monthly_category[monthly_category['user_id'] == user_id].copy()
    if user_cat.empty:
        return {}

    cat_totals = user_cat.groupby('category_id')['category_spend'].sum()
    top_cats   = cat_totals.nlargest(top_n).index.tolist()

    cat_forecasts = {}
    for cat in top_cats:
        cat_series = (user_cat[user_cat['category_id'] == cat].sort_values('month')['category_spend'])
        point, lower, upper = forecast_with_wma(cat_series)
        if point is not None:
            cat_forecasts[str(cat)] = {
                'forecast':    point,
                'lower':       lower,
                'upper':       upper,
                'hist_avg':    round(float(cat_series.mean()), 2),
                'data_points': len(cat_series)
            }

    return cat_forecasts

### Run Forecasts for All Users

In [10]:
all_forecasts = {}
method_counts = {'prophet': 0, 'wma': 0, 'segment_median': 0}
errors = []

all_user_ids = monthly_user['user_id'].unique()
total = len(all_user_ids)

for i, user_id in enumerate(all_user_ids):
    user_monthly = monthly_user[monthly_user['user_id'] == user_id].sort_values('month')
    n_months = len(user_monthly)
    segment = segment_map.get(user_id, None)
    next_month = get_next_month(user_monthly)

    forecast_record = {
        'user_id':    user_id,
        'forecast_month': next_month,
        'n_months_history': n_months,
        'segment':    segment,
    }

# Total Spend
    if n_months >= 4:
        try:
            pt, lo, hi, _ = forecast_with_prophet(user_monthly, 'total_spend')
            forecast_record.update({
                'spend_forecast': pt, 'spend_lower': lo, 'spend_upper': hi,
                'spend_method': 'prophet'
            })
            method_counts['prophet'] += 1
        except Exception as e:
            errors.append((user_id, 'spend_prophet', str(e)))
            pt, lo, hi = forecast_with_wma(user_monthly['total_spend'])
            forecast_record.update({
                'spend_forecast': pt or 0, 'spend_lower': lo or 0, 'spend_upper': hi or 0,
                'spend_method': 'wma_fallback'
            })

    elif n_months >= 2:
        pt, lo, hi = forecast_with_wma(user_monthly['total_spend'])
        forecast_record.update({
            'spend_forecast': pt or 0, 'spend_lower': lo or 0, 'spend_upper': hi or 0,
            'spend_method': 'wma'
        })
        method_counts['wma'] += 1

    else:
        seg_med = segment_medians.loc[segment, 'total_spend'] \
                  if segment in segment_medians.index else global_medians['total_spend']
        forecast_record.update({
            'spend_forecast': round(float(seg_med), 2),
            'spend_lower':    round(float(seg_med) * 0.8, 2),
            'spend_upper':    round(float(seg_med) * 1.2, 2),
            'spend_method':   'segment_median'
        })
        method_counts['segment_median'] += 1

# Transactions
    if n_months >= 4:
        try:
            pt, lo, hi, _ = forecast_with_prophet(user_monthly, 'transaction_count')
            forecast_record.update({
                'count_forecast': int(round(pt)),
                'count_lower':    int(round(lo)),
                'count_upper':    int(round(hi)),
                'count_method':   'prophet'
            })
        except:
            pt, lo, hi = forecast_with_wma(user_monthly['transaction_count'])
            forecast_record.update({
                'count_forecast': int(round(pt or 0)),
                'count_lower':    int(round(lo or 0)),
                'count_upper':    int(round(hi or 0)),
                'count_method':   'wma_fallback'
            })
    else:
        pt, lo, hi = forecast_with_wma(user_monthly['transaction_count'])
        if pt is None:
            seg_med = segment_medians.loc[segment, 'transaction_count'] \
                      if segment in segment_medians.index else global_medians['transaction_count']
            pt, lo, hi = float(seg_med), float(seg_med)*0.8, float(seg_med)*1.2
        forecast_record.update({
            'count_forecast': int(round(pt)),
            'count_lower':    int(round(lo)),
            'count_upper':    int(round(hi)),
            'count_method':   'wma'
        })

# Avg transaction amount
    spend_f = forecast_record.get('spend_forecast', 0)
    count_f = forecast_record.get('count_forecast', 1)
    forecast_record['avg_amount_forecast'] = round(spend_f / max(count_f, 1), 2)

# Historical context (for chatbot comparison)
    forecast_record['hist_avg_spend'] = round(float(user_monthly['total_spend'].mean()), 2)
    forecast_record['hist_avg_count'] = round(float(user_monthly['transaction_count'].mean()), 1)
    forecast_record['last_month_spend'] = round(float(user_monthly['total_spend'].iloc[-1]), 2)
    forecast_record['last_month_count'] = int(user_monthly['transaction_count'].iloc[-1])

# Category forecasts
    forecast_record['category_forecasts'] = forecast_categories(user_id, monthly_category)

    all_forecasts[user_id] = forecast_record

print(f'\nForecasts complete for {len(all_forecasts)} users')
print(f'Prophet: {method_counts["prophet"]} users')
print(f'Moving Average: {method_counts["wma"]} users')
print(f'Segment Median: {method_counts["segment_median"]} users')
if errors:
    print(f'   Errors caught and handled: {len(errors)}')

22:20:30 - cmdstanpy - INFO - Chain [1] start processing


22:20:30 - cmdstanpy - INFO - Chain [1] done processing
22:20:31 - cmdstanpy - INFO - Chain [1] start processing
22:20:31 - cmdstanpy - INFO - Chain [1] done processing
22:20:31 - cmdstanpy - INFO - Chain [1] start processing
22:20:31 - cmdstanpy - INFO - Chain [1] done processing
22:20:31 - cmdstanpy - INFO - Chain [1] start processing
22:20:31 - cmdstanpy - INFO - Chain [1] done processing
22:20:31 - cmdstanpy - INFO - Chain [1] start processing
22:20:31 - cmdstanpy - INFO - Chain [1] done processing
22:20:31 - cmdstanpy - INFO - Chain [1] start processing
22:20:31 - cmdstanpy - INFO - Chain [1] done processing
22:20:31 - cmdstanpy - INFO - Chain [1] start processing
22:20:31 - cmdstanpy - INFO - Chain [1] done processing
22:20:31 - cmdstanpy - INFO - Chain [1] start processing
22:20:31 - cmdstanpy - INFO - Chain [1] done processing
22:20:31 - cmdstanpy - INFO - Chain [1] start processing
22:20:31 - cmdstanpy - INFO - Chain [1] done processing
22:20:31 - cmdstanpy - INFO - Chain [1] 


Forecasts complete for 306 users
Prophet: 279 users
Moving Average: 21 users
Segment Median: 6 users


### Add Forecast Direction & Confidence

Confidence based on data quantity and method

In [11]:
for user_id, rec in all_forecasts.items():
    hist_avg = rec['hist_avg_spend']
    forecast = rec['spend_forecast']

    if hist_avg > 0:
        pct_change = (forecast - hist_avg) / hist_avg * 100
    else:
        pct_change = 0

    rec['spend_pct_change_vs_hist'] = round(pct_change, 1)

    if pct_change > 10:
        rec['spend_direction'] = 'increasing'
    elif pct_change < -10:
        rec['spend_direction'] = 'decreasing'
    else:
        rec['spend_direction'] = 'stable'

    n = rec['n_months_history']
    method = rec['spend_method']
    if method == 'prophet' and n >= 6:
        rec['confidence'] = 'high'
    elif method in ('prophet', 'wma') and n >= 3:
        rec['confidence'] = 'medium'
    else:
        rec['confidence'] = 'low'

    last = rec['last_month_spend']
    if last > 0:
        rec['spend_vs_last_month_pct'] = round((forecast - last) / last * 100, 1)
    else:
        rec['spend_vs_last_month_pct'] = 0

directions  = pd.Series({uid: r['spend_direction'] for uid, r in all_forecasts.items()})
confidences = pd.Series({uid: r['confidence']      for uid, r in all_forecasts.items()})
print(f'\nSpend direction: {directions.value_counts().to_dict()}')
print(f'Confidence: {confidences.value_counts().to_dict()}')


Spend direction: {'decreasing': 125, 'increasing': 96, 'stable': 85}
Confidence: {'high': 262, 'low': 27, 'medium': 17}


In [ ]:
print(all_forecasts.items()) ## no account_id

dict_items([('USR000Ux', {'user_id': 'USR000Ux', 'forecast_month': Timestamp('2025-11-01 00:00:00'), 'n_months_history': 2, 'segment': 'Single-Tasker', 'spend_forecast': 2000.0, 'spend_lower': 1600.0, 'spend_upper': 2400.0, 'spend_method': 'wma', 'count_forecast': 1, 'count_lower': 1, 'count_upper': 1, 'count_method': 'wma', 'avg_amount_forecast': 2000.0, 'hist_avg_spend': 2000.0, 'hist_avg_count': 1.0, 'last_month_spend': 2000.0, 'last_month_count': 1, 'category_forecasts': {'Other': {'forecast': 2000.0, 'lower': 1600.0, 'upper': 2400.0, 'hist_avg': 2000.0, 'data_points': 2}}, 'spend_pct_change_vs_hist': 0.0, 'spend_direction': 'stable', 'confidence': 'low', 'spend_vs_last_month_pct': 0.0}), ('USR001YL', {'user_id': 'USR001YL', 'forecast_month': Timestamp('2025-02-01 00:00:00'), 'n_months_history': 22, 'segment': 'Consistent Weekly', 'spend_forecast': 12843.18, 'spend_lower': 0, 'spend_upper': 70284.56, 'spend_method': 'prophet', 'count_forecast': 2, 'count_lower': 1, 'count_upper': 4

#### Export to CSV

In [13]:
export_rows = []
for uid, rec in all_forecasts.items():
    row = {
        'user_id': uid,
        'forecast_month': rec['forecast_month'].strftime('%Y-%m'),
        'segment': rec['segment'],
        'n_months_history': rec['n_months_history'],

        # Spend
        'spend_forecast': rec['spend_forecast'],
        'spend_lower': rec['spend_lower'],
        'spend_upper': rec['spend_upper'],
        'spend_method': rec['spend_method'],
        'spend_direction': rec['spend_direction'],
        'spend_pct_change_vs_hist': rec['spend_pct_change_vs_hist'],
        'spend_vs_last_month_pct': rec['spend_vs_last_month_pct'],

        # Count
        'count_forecast': rec['count_forecast'],
        'count_lower': rec['count_lower'],
        'count_upper': rec['count_upper'],
        'count_method': rec['count_method'],

        # Avg amount
        'avg_amount_forecast': rec['avg_amount_forecast'],

        # Historical context
        'hist_avg_spend': rec['hist_avg_spend'],
        'hist_avg_count': rec['hist_avg_count'],
        'last_month_spend': rec['last_month_spend'],
        'last_month_count': rec['last_month_count'],

        # Confidence
        'confidence': rec['confidence'],
    }

    # Flatten top 3 category forecasts
    for i, (cat, vals) in enumerate(list(rec['category_forecasts'].items())[:3], 1):
        row[f'cat{i}_name'] = cat
        row[f'cat{i}_forecast'] = vals['forecast']
        row[f'cat{i}_hist_avg'] = vals['hist_avg']

    export_rows.append(row)

forecasts_csv = pd.DataFrame(export_rows).set_index('user_id')
forecasts_csv.to_csv('outputs/forecasts.csv')

display(forecasts_csv[[
    'forecast_month', 'segment', 'spend_forecast', 'spend_direction',
    'count_forecast', 'avg_amount_forecast', 'confidence'
]].head(10))

,forecast_month,segment,spend_forecast,spend_direction,count_forecast,avg_amount_forecast,confidence
user_id,,,,,,,
USR000Ux,2025-11,Single-Tasker,2000.00,stable,1,2000.00,low
USR001YL,2025-02,Consistent Weekly,12843.18,increasing,2,6421.59,high
USR001ul,2025-11,Single-Tasker,25.25,decreasing,2,12.62,high
USR002SI,2025-11,Healthy Active User,3980.95,stable,6,663.49,high
USR006To,2025-11,Healthy Active User,1542.83,decreasing,3,514.28,low
USR007mU,2025-01,Single-Tasker,1213.59,increasing,3,404.53,high
USR009qR,2025-02,Single-Tasker,435.75,decreasing,2,217.88,high
USR013pA,2025-01,Single-Tasker,0.00,decreasing,3,0.00,high
USR017aL,2025-11,Single-Tasker,534.40,decreasing,5,106.88,high
